# 01 · Data import

**Goal:** load the Lending Club loans, keep only those that are useful to model credit risk (**PD**, **EAD** and **LGD**) and split them into a **training** set (70 %) and a **validation** set (30 %), held out to evaluate the models at the end.

Every decision is based on figures computed in this same notebook. Each one appears in a **Decision Dx** box stating *what* is done, *why*, and *which alternatives were discarded*.

> **Note on names:** column names, category values and code identifiers are kept in Spanish, as in the data files and the API used across the project. The data dictionary in section 7.2 explains every column.

<a id="s0"></a>

## Decision log

| # | Decision | Main reason |
|---|---|---|
| D1 | Use the public Lending Club dataset (Kaggle, CC0) without versioning it in Git | Free licence, but 1.7 GB does not fit in a repository |
| D2 | Read the CSV in chunks and only the required columns | The full file does not fit in the memory of a normal computer |
| D3 | Remove the totals rows | They are not loans: leftovers from merging the quarterly CSVs |
| D4 | Keep only finished loans | A loan still in progress has no outcome yet |
| D5 | Use only cohorts with ≥ 95 % finished loans | Avoids the bias of recent cohorts |
| D6 | Select columns by role and set apart those that cause *leakage* | Only variables known when the loan is issued can be used |
| D7 | Start in 2012 | Before that, model variables are missing (structural nulls) |
| D8 | Individual applications only | In joint applications, income and DTI mean something else |
| D9 | Drop columns with > 80 % nulls | They do not provide reliable information |
| D10 | Rename columns to Spanish | Consistency with the rest of the project (notebooks 02–08 and API) |
| D11 | Do not clean values here (only convert dates) | Data quality is handled in notebook 02 |
| D12 | Stratified 70/30 split with a fixed seed | Reproducible and with the same default rate in both sets |

## Table of contents

- [Decision log](#s0)
- [1 · Import libraries and settings](#s1)
- [2 · D1 · Data source](#s2)
- [3 · D2 · Reading in chunks](#s3)
  - [3.1 · First read: minimum columns](#s3-1)
- [4 · D3 · Totals rows](#s4)
- [5 · D4 · Loan status](#s5)
- [6 · D5 · Cohorts](#s6)
- [7 · D6 · Column selection](#s7)
  - [7.1 · Classification of the 151 columns](#s7-1)
  - [7.2 · Dictionary of the selected variables](#s7-2)
  - [7.3 · Second read: selected columns](#s7-3)
- [8 · D7 · Start year](#s8)
- [9 · D8 · Application type](#s9)
- [10 · D9 · Almost empty columns](#s10)
- [11 · D10 · Spanish names](#s11)
- [12 · D11 · Data types](#s12)
- [13 · D12 · Training / validation split](#s13)
- [14 · Save the datasets](#s14)
- [15 · Summary: row funnel](#s15)
- [16 · Next steps](#s16)

> Each section starts with a **blue band**. Below every result there is an explanation of **what is done** and **what we see**.

<a id="s1"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">1 · IMPORT LIBRARIES AND SETTINGS</span>

</div>

In [ ]:
# --- Libraries -------------------------------------------------------------
from pathlib import Path                               # OS-independent file paths
import numpy as np                                     # numerical computing
import pandas as pd                                    # tables (DataFrames)
from sklearn.model_selection import train_test_split   # training / validation split

# --- Display options -------------------------------------------------------
pd.set_option('display.float_format', lambda x: '%.3f' % x)  # no scientific notation
pd.set_option('display.max_columns', 60)                     # show every column of a table
pd.set_option('display.max_rows', 200)                       # show long tables in full (e.g. the 151 columns)
np.set_printoptions(suppress=True)

print("✅ Libraries imported successfully")

**What is done:** the notebook libraries are imported and table display is configured (no scientific notation, all columns and rows visible).

**What we see:** the message confirms that every library is installed in the environment.

In [ ]:
# --- Paths (relative to this notebook's folder: 03_notebooks/en) -----------
RUTA_DATOS = Path('../../02_datos')
RUTA_ORIGINAL = RUTA_DATOS / '00_Originales' / 'accepted_2007_to_2018Q4.csv'
RUTA_ENTRENAMIENTO = RUTA_DATOS / '03_Entrenamiento' / 'train.pkl'
RUTA_VALIDACION = RUTA_DATOS / '02_Validacion' / 'validacion.pkl'

# --- Decision parameters (each one is justified below) ---------------------
TAM_BLOQUE = 200_000        # D2: rows read in each chunk
UMBRAL_TERMINADOS = 0.95    # D5: minimum share of finished loans to accept a cohort
ANIO_INICIO = 2012          # D7: first issue year that is kept
UMBRAL_NULOS = 0.80         # D9: maximum share of nulls tolerated in a column
PROP_VALIDACION = 0.30      # D12: share held out for validation
SEMILLA = 42                # D12: fixed seed so that the split is reproducible

# Dictionary that records how many rows remain after each step (funnel at the end)
embudo = {}

<a id="s2"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">2 · D1 · DATA SOURCE</span>

</div>

> **Decision D1 · Use the public dataset *All Lending Club loan data* (Kaggle, CC0 licence) and do not upload it to Git.**
> - **What:** the file `accepted_2007_to_2018Q4.csv`, with the loans **issued** between 2007 and 2018. The rejected-applications file is not used because it has no repayment outcome, so it is useless for PD, EAD and LGD.
> - **Why:** real data from a peer-to-peer lending platform under a public domain licence (CC0), so it can be used and shared freely.
> - **Why not in Git:** it takes 1.7 GB. It is excluded in `.gitignore`.
> - **How to reproduce it:** download the dataset from Kaggle (author *wordsforthewise*), unzip the accepted-loans file and save it in `02_datos/00_Originales/`.

In [ ]:
# Check that the file exists before continuing (if not, explain how to get it)
if not RUTA_ORIGINAL.exists():
    raise FileNotFoundError(
        f"{RUTA_ORIGINAL} not found. Download 'All Lending Club loan data' from Kaggle "
        "and save accepted_2007_to_2018Q4.csv in 02_datos/00_Originales/"
    )

# File size in GB
print(f"File size: {RUTA_ORIGINAL.stat().st_size / 1e9:.2f} GB")

# Read only the header (nrows=0) to get the columns without loading any data
cabecera = pd.read_csv(RUTA_ORIGINAL, nrows=0)
print(f"Number of columns: {cabecera.shape[1]}")

**What is done:** we check that the original file exists and read **only the header** (`nrows=0`), without loading any data, to get its columns.

**What we see:** the file takes **1.68 GB** and has **151 columns**. Given its size, it should not be loaded in one go (decision D2).

<a id="s3"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">3 · D2 · READING IN CHUNKS</span>

</div>

> **Decision D2 · Read the CSV in chunks of 200,000 rows, loading only the required columns and filtering each chunk before keeping it.**
> - **Why:** the file has 2.26 million rows and 151 columns. Loaded in full in pandas it takes several GB of RAM (on a 4 GB computer not even 40 columns fit). Reading in chunks, only the rows that pass the filter are kept in memory.
> - **How:** the function `leer_por_bloques` is used twice. First, 4 columns of every row are read to decide **which rows** are useful (D3–D5). Then the columns chosen in D6 are read, **only** for those rows.
> - **Discarded alternatives:** loading everything at once (runs out of memory) or converting it to Parquet first (needs extra libraries; can be considered later if reading turns out to be slow).

In [ ]:
def leer_por_bloques(columnas, filtro=None, tam_bloque=TAM_BLOQUE):
    '''Reads the original CSV in chunks and returns a single DataFrame.

    columnas : list of columns to read (the rest is not even loaded)
    filtro   : optional function that takes a chunk and returns a boolean mask
               with the rows to keep
    '''
    bloques = []                     # filtered chunks are accumulated here
    lector = pd.read_csv(
        RUTA_ORIGINAL,
        usecols=columnas,            # only the requested columns
        chunksize=tam_bloque,        # returns an iterator of chunks instead of a single DataFrame
        dtype={'id': str},           # 'id' as text: the totals rows have text in that column (D3)
        low_memory=False,            # infer each column's type using the whole chunk
    )
    for bloque in lector:
        if filtro is not None:
            bloque = bloque[filtro(bloque)]   # keep only the rows that pass the filter
        bloques.append(bloque)
    return pd.concat(bloques)        # join every chunk (the original index is kept)

<a id="s3-1"></a>

### 3.1 · First read: the minimum columns to decide which rows are useful

In [ ]:
# First read: identifier, issue date, term and loan status
base = leer_por_bloques(['id', 'issue_d', 'term', 'loan_status'])
embudo['1. Original file'] = len(base)
print(f"Rows read: {len(base):,}")

**What is done:** first read in chunks: only 4 columns (identifier, issue date, term and status), but for **every** row.

**What we see:** the file has **2,260,701 rows**. These 4 columns are enough to decide which rows are useful (D3–D5) before reading the rest.

<a id="s4"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">4 · D3 · TOTALS ROWS</span>

</div>

In [ ]:
# Rows without loan status: what do they contain?
filas_sin_estado = base[base['loan_status'].isna()]
print(f"Rows without status: {len(filas_sin_estado)}")
filas_sin_estado.head(6)

**What is done:** we look for the rows without loan status and show the first ones.

**What we see:** there are **33 rows** without status. They are not loans: the `id` column contains texts such as *"Total amount funded in policy code 1: …"* and every other column is empty.

> **Decision D3 · Remove the rows without `loan_status`.**
> - **Why:** they are not loans but totals lines (*"Total amount funded in policy code…"*) left over from merging the quarterly CSVs that Lending Club used to publish.

In [ ]:
# Remove the totals rows
base = base.dropna(subset=['loan_status'])

# Without those rows, the identifier can be converted to integer
base['id'] = base['id'].astype('int64')

embudo['2. Without totals rows (D3)'] = len(base)
print(f"Remaining rows: {len(base):,}")

**What is done:** the 33 totals rows are removed and the identifier is converted to integer (it now only contains numbers).

**What we see:** **2,260,668 rows** remain.

<a id="s5"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">5 · D4 · LOAN STATUS</span>

</div>

**Variable analysed: `loan_status` (in Spanish, `estado`)** · status of the loan at the dataset cut-off date (end of 2018):
- `Fully Paid`: repaid in full.
- `Charged Off`: written off (**default**): Lending Club no longer expects payments.
- `Current`: in progress and up to date.
- `In Grace Period`: up to 15 days late (grace period).
- `Late (16-30 days)` / `Late (31-120 days)`: payment overdue by that number of days.
- `Default`: more than 120 days late, not yet written off.
- `Does not meet the credit policy. Status: …`: old loans issued under criteria that no longer meet the current credit policy; their final status appears after the colon.

In [ ]:
# Statuses that mean the loan has finished (outcome known)
ESTADOS_TERMINADOS = [
    'Fully Paid',
    'Charged Off',
    'Does not meet the credit policy. Status:Fully Paid',
    'Does not meet the credit policy. Status:Charged Off',
]
# Finished statuses that count as default (same criterion as notebook 04)
ESTADOS_IMPAGO = ['Charged Off', 'Does not meet the credit policy. Status:Charged Off']

# Status table: number of loans, percentage and whether the outcome is known
tabla_estados = base['loan_status'].value_counts().to_frame('n_prestamos')
tabla_estados['%'] = tabla_estados['n_prestamos'] / len(base) * 100
tabla_estados['terminado'] = tabla_estados.index.isin(ESTADOS_TERMINADOS)
tabla_estados['impago'] = tabla_estados.index.isin(ESTADOS_IMPAGO)
tabla_estados

**What is done:** table with the number (`n_prestamos`) and % of loans in each status. The columns `terminado` (finished) and `impago` (default) show whether the status is final and whether it counts as a default.

**What we see:** **Fully Paid** (47.6 %) and **Charged Off** (11.9 %) are the main finished statuses. **Current** (38.9 %) are loans still in progress, with no known outcome. Late payments (`Late`, `In Grace Period`, `Default`) add up to about 1.5 %.

> **Decision D4 · Keep only finished loans**: `Fully Paid`, `Charged Off` and their *"Does not meet the credit policy"* variants.
> - **Why:** PD estimates whether a loan **will end** in default. A `Current`, `Late` or `In Grace Period` loan can still end either way. Labelling it as "no default" would make the model learn from wrong labels. The same applies to EAD and LGD: they need the final amount repaid and recovered.
> - `Default` is not closed either: it is a delay of more than 120 days that has not yet been written off.
> - **Discarded alternative:** counting `Late (31-120 days)` and `Default` as default. It is common in banking (90-days-past-due definition), but it would mix in loans whose final outcome we do not know.

In [ ]:
# Flag each loan: has it finished? has it finished in default?
base['terminado'] = base['loan_status'].isin(ESTADOS_TERMINADOS)
base['impago'] = base['loan_status'].isin(ESTADOS_IMPAGO)

embudo['3. Finished loans only (D4)'] = int(base['terminado'].sum())
print(f"Finished loans: {base['terminado'].sum():,}")

**What is done:** each loan is flagged according to whether it has finished and whether it finished in default.

**What we see:** there are **1,348,059 finished loans** (59.6 % of the file).

<a id="s6"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">6 · D5 · COHORTS</span>

</div>

A **cohort** is the group of loans issued in the same year and with the same term (36 or 60 months). The dataset ends in late 2018, so many recent cohorts have not finished yet.

**Variables analysed:**
- `issue_d` (in Spanish, `fecha_concesion`): month and year in which the loan was issued (format `Dec-2015`).
- `term` (in Spanish, `num_cuotas`): loan term: 36 or 60 monthly instalments.

In [ ]:
# Issue year ('issue_d' has the format 'Dec-2015')
base['anio'] = pd.to_datetime(base['issue_d'], format='%b-%Y').dt.year

# Term without the leading space (' 36 months' -> '36 months')
base['plazo'] = base['term'].str.strip()

# By cohort (term, year): number of loans and share of finished loans
cohortes = base.groupby(['plazo', 'anio']).agg(
    n_prestamos=('id', 'size'),
    pct_terminados=('terminado', 'mean'),
)
# Default rate computed ONLY among the finished loans of each cohort
cohortes['tasa_impago_terminados'] = base[base['terminado']].groupby(['plazo', 'anio'])['impago'].mean()

# A cohort is valid if almost all of its loans have finished
cohortes['valida'] = cohortes['pct_terminados'] >= UMBRAL_TERMINADOS
cohortes

**What is done:** for each **cohort** (term × issue year) we compute the number of loans, the share of finished loans (`pct_terminados`, from 0 to 1), the default rate **only among the finished ones** (`tasa_impago_terminados`) and whether the cohort is valid (`valida`: at least 95 % finished).

**What we see:** 36-month cohorts are complete up to 2015 (≥ 99.9 %) and drop afterwards (72 % in 2016). 60-month cohorts, only up to 2013 (83 % in 2014). In the incomplete 60-month cohorts the **apparent** default rate shoots up (31–36 %) because defaults arrive earlier than full repayments.

> **Decision D5 · Use only the cohorts with at least 95 % finished loans.**
> - **Why:** in a recent cohort, the loans that have already finished are not representative. A default usually happens before the last payment of a loan that is repaid normally, so among the finished loans there are "too many" defaults (**selection bias**). The table shows it: in the 2015 60-month loans only 67 % have finished and their apparent default rate is 36 %, versus 25 % in the 2013 cohort, which is already complete.
> - **Result:** 36-month loans are kept up to 2015 and 60-month loans up to 2013 (column `valida`).
> - **Discarded alternatives:** using every finished loan (selection bias) or survival models, which take unfinished loans into account (more complex and out of the scope of this project).

In [ ]:
# List of valid cohorts as (term, year) pairs
cohortes_validas = cohortes[cohortes['valida']].index.tolist()
print("Valid cohorts:")
for plazo, anio in cohortes_validas:
    print(f"  {plazo} · {anio}")

# Does each loan belong to a valid cohort?
base['en_cohorte_valida'] = pd.MultiIndex.from_arrays([base['plazo'], base['anio']]).isin(cohortes_validas)

# Identifiers of the loans that pass D3, D4 and D5
ids_seleccionados = base.loc[base['terminado'] & base['en_cohorte_valida'], 'id']
embudo['4. In cohorts with ≥ 95 % finished (D5)'] = len(ids_seleccionados)
print(f"\nSelected loans: {len(ids_seleccionados):,}")

# Free memory: the 'base' table is no longer needed (only the selected identifiers)
del base

**What is done:** the valid cohorts are listed and the identifiers of the loans that pass D3, D4 and D5 are kept. Then the memory used by the `base` table is freed.

**What we see:** there are **13 valid cohorts** (36 months from 2007 to 2015 and 60 months from 2010 to 2013) with **676,302 loans**.

<a id="s7"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">7 · D6 · COLUMN SELECTION</span>

</div>

The file has 151 columns. Each one is classified by its **role**:

- **Identifier** of the loan.
- **Origination**: what is known **when applying** for the loan. They are the only ones that can be model variables.
- **Target**: what happens **after** the loan is issued. They are only used to build what we want to predict (PD, EAD, LGD).
- **Auxiliary**: only to filter or split the data.
- **Discarded**: the rest, with its reason.

> **What is information leakage?** Training with data that is not known at prediction time. For example, `total_pymnt` (total paid by the borrower) reveals whether the loan was repaid. A model using it would have a very high AUC in the notebook and be useless in the demo, where a client is scored **before** the loan is issued.

In [ ]:
# Columns that are kept: Lending Club name -> (Spanish name, role)
COLUMNAS = {
    # --- Identifiers -------------------------------------------------------
    'id':                       ('id_cliente',                  'Identifier'),
    'member_id':                ('id_prestamo',                 'Identifier'),
    # --- Origination: application and loan data (14) -----------------------
    'emp_title':                ('empleo',                      'Origination'),
    'emp_length':               ('antigüedad_empleo',           'Origination'),
    'annual_inc':               ('ingresos',                    'Origination'),
    'verification_status':      ('ingresos_verificados',        'Origination'),
    'home_ownership':           ('vivienda',                    'Origination'),
    'dti':                      ('dti',                         'Origination'),
    'desc':                     ('descripcion',                 'Origination'),
    'purpose':                  ('finalidad',                   'Origination'),
    'funded_amnt':              ('principal',                   'Origination'),
    'term':                     ('num_cuotas',                  'Origination'),
    'int_rate':                 ('tipo_interes',                'Origination'),
    'installment':              ('imp_cuota',                   'Origination'),
    'grade':                    ('rating',                      'Origination'),
    'sub_grade':                ('subrating',                   'Origination'),
    # --- Origination: credit bureau history (16) ---------------------------
    'fico_range_low':           ('fico_min',                    'Origination'),
    'fico_range_high':          ('fico_max',                    'Origination'),
    'earliest_cr_line':         ('fecha_primer_credito',        'Origination'),
    'open_acc':                 ('num_lineas_credito',          'Origination'),
    'total_acc':                ('num_lineas_credito_total',    'Origination'),
    'mort_acc':                 ('num_hipotecas',               'Origination'),
    'revol_bal':                ('saldo_revolving',             'Origination'),
    'revol_util':               ('porc_uso_revolving',          'Origination'),
    'bc_util':                  ('porc_uso_tarjetas',           'Origination'),
    'percent_bc_gt_75':         ('porc_tarjetas_75p',           'Origination'),
    'inq_last_6mths':           ('num_consultas_6meses',        'Origination'),
    'delinq_2yrs':              ('num_retrasos_2anios',         'Origination'),
    'mths_since_last_delinq':   ('num_meses_desde_ult_retraso', 'Origination'),
    'pub_rec':                  ('num_derogatorios',            'Origination'),
    'pub_rec_bankruptcies':     ('num_quiebras',                'Origination'),
    'chargeoff_within_12_mths': ('num_cancelaciones_12meses',   'Origination'),
    # --- Target: only known AFTER the loan is issued -----------------------
    'loan_status':              ('estado',                      'Target (post-issuance)'),
    'total_rec_prncp':          ('imp_amortizado',              'Target (post-issuance)'),
    'recoveries':               ('imp_recuperado',              'Target (post-issuance)'),
    # --- Auxiliary: only to filter or split (not model variables) ----------
    'issue_d':                  ('fecha_concesion',             'Auxiliary'),
    'application_type':         ('tipo_solicitud',              'Auxiliary'),
}

# Check: every chosen column exists in the file
faltan = set(COLUMNAS) - set(cabecera.columns)
assert not faltan, f"Columns that do not exist in the file: {faltan}"
print(f"Selected columns: {len(COLUMNAS)}")

**What is done:** the dictionary of kept columns is defined: original name → (Spanish name, role), and we check that all of them exist in the file.

**What we see:** **37 columns** are selected and all of them exist. The meaning of each one is in the dictionary in section 7.2.

<a id="s7-1"></a>

### 7.1 · Classification of the 151 columns of the file

In [ ]:
# Rules to classify the discarded columns
# Known after the loan is issued (would cause leakage): payments, balances, refinancing, settlements...
POST_CONCESION_PREFIJOS = ('out_prncp', 'total_pymnt', 'total_rec_', 'last_', 'next_pymnt',
                           'hardship', 'settlement', 'debt_settlement', 'payment_plan',
                           'deferral', 'orig_projected')
POST_CONCESION = {'collection_recovery_fee', 'pymnt_plan'}
# Free text, links and geography
TEXTO_GEOGRAFIA = {'url', 'title', 'zip_code', 'addr_state'}
# Redundant or related to the platform's operations (they do not describe the client)
REDUNDANTES = {'loan_amnt', 'funded_amnt_inv', 'policy_code', 'initial_list_status', 'disbursement_method'}

def clasificar_columna(col):
    '''Returns (role, reason) for a column of the original file.'''
    if col in COLUMNAS:
        return COLUMNAS[col][1], 'Selected'
    if col.startswith('sec_app_') or col.endswith('_joint'):
        return 'Discarded', 'Second applicant (joint applications, see D8)'
    if col.startswith(POST_CONCESION_PREFIJOS) or col in POST_CONCESION:
        return 'Discarded', 'Known after issuance (leakage)'
    if col in TEXTO_GEOGRAFIA:
        return 'Discarded', 'Free text, link or geography'
    if col in REDUNDANTES:
        return 'Discarded', 'Redundant or platform operations'
    return 'Discarded', 'Other origination variable (parsimony)'

# Classify the 151 columns of the file
clasificacion = pd.DataFrame(
    [(col, *clasificar_columna(col)) for col in cabecera.columns],
    columns=['columna', 'grupo', 'motivo'],
)

# Summary: number of columns by role and reason
clasificacion.groupby(['grupo', 'motivo']).size().to_frame('n_columnas')

**What is done:** the 151 columns of the file are classified with a few rules (prefixes and lists) and we summarise how many fall into each role (`grupo`) and reason (`motivo`).

**What we see:** 30 origination, 3 target, 2 auxiliary and 2 identifiers. Among the discarded ones, **36 are due to *leakage*** (information known after issuance), 53 to parsimony, 16 belong to the second applicant, 5 are redundant and 4 are text or geography.

In [ ]:
# Full table (one row per column) to review each case
clasificacion

**What is done:** the full classification is shown: one row per column of the file, with its role and reason.

**What we see:** it allows checking case by case that no *leakage* column has slipped in among the selected ones.

<a id="s7-2"></a>

### 7.2 · Dictionary of the selected variables

| Spanish name | Original (Lending Club) | Role | Meaning |
|---|---|---|---|
| `id_cliente` | `id` | Identifier | Unique identifier of the **loan** in Lending Club (despite its name, it identifies the loan). |
| `id_prestamo` | `member_id` | Identifier | Client identifier. It is 100 % empty (anonymised) and dropped in D9. |
| `empleo` | `emp_title` | Origination | Job title stated by the borrower (free text). |
| `antigüedad_empleo` | `emp_length` | Origination | Years in the current job, from `< 1 year` to `10+ years`. |
| `ingresos` | `annual_inc` | Origination | Annual income stated by the borrower ($). |
| `ingresos_verificados` | `verification_status` | Origination | Whether Lending Club checked the income: `Verified`, `Source Verified` (the source was checked) or `Not Verified`. |
| `vivienda` | `home_ownership` | Origination | Housing situation: `RENT`, `MORTGAGE` (owned with a mortgage), `OWN` (owned outright) or other rare values. |
| `dti` | `dti` | Origination | *Debt-to-income*: monthly payments of all debts (excluding the mortgage and this loan) / monthly income, in %. |
| `descripcion` | `desc` | Origination | Free text where the borrower explains the loan. Dropped in D9 (> 80 % nulls). |
| `finalidad` | `purpose` | Origination | Stated purpose: debt consolidation, credit card payoff, home improvement, car… |
| `principal` | `funded_amnt` | Origination | Amount lent ($). |
| `num_cuotas` | `term` | Origination | Term: 36 or 60 monthly instalments. |
| `tipo_interes` | `int_rate` | Origination | Annual interest rate (%) assigned by Lending Club according to risk. |
| `imp_cuota` | `installment` | Origination | Monthly instalment paid by the borrower ($). |
| `rating` | `grade` | Origination | Lending Club risk grade: from `A` (lowest risk) to `G` (highest risk). |
| `subrating` | `sub_grade` | Origination | Sub-level within the grade, from `A1` to `G5` (5 levels per grade). |
| `fico_min` / `fico_max` | `fico_range_low` / `fico_range_high` | Origination | Lower and upper bounds of the borrower's **FICO score** range (300 to 850; higher = better history). |
| `fecha_primer_credito` | `earliest_cr_line` | Origination | Date of the borrower's first credit line: shows how old their credit history is. |
| `num_lineas_credito` | `open_acc` | Origination | Number of **open** credit lines (cards, loans…). |
| `num_lineas_credito_total` | `total_acc` | Origination | **Total** number of credit lines in the history, open and closed. |
| `num_hipotecas` | `mort_acc` | Origination | Number of mortgage accounts in the credit report. |
| `saldo_revolving` | `revol_bal` | Origination | Outstanding *revolving* credit balance ($): credit that can be used, repaid and used again up to a limit (cards, credit lines). |
| `porc_uso_revolving` | `revol_util` | Origination | % of the total *revolving* limit in use (balance / limit). |
| `porc_uso_tarjetas` | `bc_util` | Origination | % of the limit of their **bankcards** in use. |
| `porc_tarjetas_75p` | `percent_bc_gt_75` | Origination | % of their bankcards above 75 % of their limit. |
| `num_consultas_6meses` | `inq_last_6mths` | Origination | Number of credit inquiries in the last 6 months (each new credit application creates one). |
| `num_retrasos_2anios` | `delinq_2yrs` | Origination | Number of payments more than 30 days late in the last 2 years. |
| `num_meses_desde_ult_retraso` | `mths_since_last_delinq` | Origination | Months since the last late payment (empty if none is recorded). |
| `num_derogatorios` | `pub_rec` | Origination | Number of derogatory public records (judgments, liens, bankruptcies…). |
| `num_quiebras` | `pub_rec_bankruptcies` | Origination | Number of bankruptcies in public records. |
| `num_cancelaciones_12meses` | `chargeoff_within_12_mths` | Origination | Number of debts a creditor wrote off in the last 12 months. |
| `estado` | `loan_status` | Target | Final status of the loan (see section 5). |
| `imp_amortizado` | `total_rec_prncp` | Target | Principal repaid by the borrower ($). |
| `imp_recuperado` | `recoveries` | Target | Amount recovered after default ($). |
| `fecha_concesion` | `issue_d` | Auxiliary | Month the loan was issued. |
| `tipo_solicitud` | `application_type` | Auxiliary | Individual or joint application. Used in D8 and then dropped. |

> **Decision D6 · Keep 37 columns: 2 identifiers, 30 origination variables, 3 to build the targets and 2 auxiliary ones.**
> - **The 30 origination variables** cover the three classic blocks of credit *scoring*, all known when applying for the loan:
>   - **Repayment capacity:** income (and whether it is verified), DTI (debt / income), instalment, job and housing.
>   - **Loan conditions:** principal, term, interest rate, purpose, grade and sub-grade (5 levels within each grade).
>   - **Credit bureau history:** FICO score, length of credit history (from the date of the first credit line), open credit lines, card and *revolving* credit utilisation, recent inquiries, late payments, derogatory public records and bankruptcies.
> - **`principal` = `funded_amnt`** (amount actually lent), which is the right basis to compute exposure (EAD). `loan_amnt` is the *requested* amount and is discarded as redundant.
> - **Why not more:** every model variable has to be asked for in the demo form, so a small and interpretable set is preferred. The remaining origination variables can be explored in a future version.
> - **Geography out** (`addr_state`, `zip_code`): it would add dozens of categories and can act as a proxy for protected characteristics (risk of discrimination).
> - The 3 **target columns will never be used as input variables**.

<a id="s7-3"></a>

### 7.3 · Second read: selected columns, only for the selected loans

In [ ]:
def es_seleccionado(bloque):
    '''Mask with the loans that passed D3, D4 and D5.'''
    # pd.to_numeric(errors='coerce') turns the text of the totals rows into NaN
    return pd.to_numeric(bloque['id'], errors='coerce').isin(ids_seleccionados)

# Read only the D6 columns and only the selected rows
prestamos = leer_por_bloques(list(COLUMNAS), filtro=es_seleccionado)
prestamos['id'] = prestamos['id'].astype('int64')

print(f"Loans: {len(prestamos):,} · Columns: {prestamos.shape[1]}")
print(f"Memory used: {prestamos.memory_usage(deep=True).sum() / 1e6:,.0f} MB")

**What is done:** second read in chunks: only the 37 D6 columns and only the selected loans.

**What we see:** **676,302 loans × 37 columns** remain, taking about 721 MB of memory.

<a id="s8"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">8 · D7 · START YEAR</span>

</div>

In [ ]:
# Issue year of each selected loan
anio = pd.to_datetime(prestamos['issue_d'], format='%b-%Y').dt.year

# % of nulls per year in the origination variables (only those with some nulls)
variables_originacion = [lc for lc, (_, grupo) in COLUMNAS.items() if grupo.startswith('Origination')]
nulos_por_anio = prestamos[variables_originacion].isna().groupby(anio).mean() * 100
nulos_por_anio = nulos_por_anio.loc[:, nulos_por_anio.max() > 0]

# Transposed: one row per variable and one column per year
nulos_por_anio.T.round(1)

**What is done:** % of nulls per issue year for each origination variable (only those with some nulls). Each row is a variable and each column a year.

**What we see:** `mort_acc`, `bc_util` and `percent_bc_gt_75` are **100 % empty from 2007 to 2011** (Lending Club did not collect them yet), have ~15 % nulls in 2012 (January–March, see notebook 02) and ~1 % or less from 2013 onwards. `desc` (description) stops being filled in over the years and `mths_since_last_delinq` has ~50 % nulls every year (with a meaning: handled in notebook 02).

> **Decision D7 · Keep only the loans issued from 2012 onwards.**
> - **Why:** `mort_acc`, `percent_bc_gt_75` and `bc_util` are 100 % empty between 2007 and 2011, because Lending Club did not collect them yet. It is a **structural** null: imputing it would mean inventing the value for entire years, and the model could learn "old loan = null" instead of the real risk.
> - **Cost:** about 6 % of the loans are lost (see below). The *"Does not meet the credit policy"* loans also disappear: all of them are from 2007–2010 and were issued under a different risk policy.
> - **Discarded alternative:** keeping 2007–2011 and dropping those three variables. Useful variables would be lost for every later year.

In [ ]:
# Loans lost when starting in ANIO_INICIO
antes = anio < ANIO_INICIO
print(f"Loans before {ANIO_INICIO}: {antes.sum():,} ({antes.mean():.1%})")

# Issue year of the "Does not meet the credit policy" loans
no_cumplen = prestamos['loan_status'].str.startswith('Does not meet')
print("\n'Does not meet the credit policy' loans by year:")
print(anio[no_cumplen].value_counts().sort_index().to_string())

# Apply the filter
prestamos = prestamos[~antes]
embudo[f'5. Issued from {ANIO_INICIO} (D7)'] = len(prestamos)
print(f"\nRemaining loans: {len(prestamos):,}")

**What is done:** we count the loans issued before 2012 and the *"Does not meet the credit policy"* loans per year, and apply the D7 filter.

**What we see:** **42,535 loans (6.3 %)** are lost. The *"Does not meet the credit policy"* loans are all from 2007–2010, so they disappear with this filter. **633,767** remain.

<a id="s9"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">9 · D8 · APPLICATION TYPE</span>

</div>

**Variable analysed: `application_type`** · type of application: `Individual` (one applicant) or `Joint App` (joint application by two people, whose income and debts are assessed together).

In [ ]:
# Number of loans by application type
prestamos['application_type'].value_counts()

**What is done:** we count the loans by application type.

**What we see:** almost all are individual (**633,528**); only **239** are joint.

> **Decision D8 · Keep only individual applications.**
> - **Why:** in a joint application the risk depends on two people, and the relevant income and DTI are the combined ones (`annual_inc_joint`, `dti_joint`), which are not used. There are very few of them (see table), so removing them barely affects the volume and keeps the same meaning for each variable.

In [ ]:
# Filter and drop the auxiliary column (from here on all applications are individual)
prestamos = prestamos[prestamos['application_type'] == 'Individual'].drop(columns='application_type')
embudo['6. Individual applications only (D8)'] = len(prestamos)
print(f"Remaining loans: {len(prestamos):,}")

**What is done:** only individual applications are kept and the `application_type` column, which no longer adds anything, is dropped.

**What we see:** **633,528 loans** remain.

<a id="s10"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">10 · D9 · ALMOST EMPTY COLUMNS</span>

</div>

In [ ]:
# % of nulls of each column after the filters (only those with some nulls)
pct_nulos = prestamos.isna().mean().sort_values(ascending=False) * 100
pct_nulos[pct_nulos > 0].round(2).to_frame('% nulos')

**What is done:** % of nulls of each column after the previous filters (only those with some nulls).

**What we see:** `member_id` is 100 % empty and `desc` 85 %. The rest stay below 51 %; the nulls of those variables are analysed in notebook 02.

> **Decision D9 · Drop the columns with more than 80 % nulls**: `member_id` (100 % empty, because Lending Club anonymised it) and `desc` (free text that hardly anyone filled in).
> - `mths_since_last_delinq` (around half nulls) **is kept**: the null has a meaning (the borrower has never been late) and it will be handled in notebook 02.

In [ ]:
# Columns above the null threshold
columnas_vacias = pct_nulos[pct_nulos > UMBRAL_NULOS * 100].index.tolist()
print("Dropped columns:", columnas_vacias)
prestamos = prestamos.drop(columns=columnas_vacias)

# Check: each loan appears only once
assert prestamos['id'].is_unique, "There are duplicated identifiers"
print("✅ No duplicated identifiers")

**What is done:** the columns above 80 % nulls are dropped and we check that no loan is repeated.

**What we see:** `member_id` and `desc` are dropped, and there are no duplicated identifiers.

<a id="s11"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">11 · D10 · SPANISH NAMES</span>

</div>

> **Decision D10 · Rename the columns to Spanish.**
> - **Why:** consistency with the rest of the project. Notebooks 02–08 and the API deployed on AWS (which receives these names from the demo form) use these same names. The columns are also ordered by role (identifier, origination, targets, dates) to make the table easier to read.
> - **Discarded alternative:** keeping the original English names, which would require translating them at every step up to the API.

In [ ]:
# Rename with the D6 dictionary (pandas ignores the columns that no longer exist)
prestamos = prestamos.rename(columns={lc: es for lc, (es, _) in COLUMNAS.items()})

# Order the columns as in the D6 dictionary (grouped by role)
orden = [es for es, _ in COLUMNAS.values() if es in prestamos.columns]
prestamos = prestamos[orden]
print(list(prestamos.columns))

**What is done:** the columns are renamed to Spanish with the D6 dictionary and ordered by role.

**What we see:** **34 columns** remain: identifier, 29 origination, 3 target and the issue date.

<a id="s12"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">12 · D11 · DATA TYPES</span>

</div>

> **Decision D11 · Do not clean values in this notebook; only convert the two dates.**
> - **Why:** each notebook has a single responsibility. Texts such as `' 36 months'` or `'10+ years'`, nulls and anomalous values are handled in data-quality notebook 02. Dates are converted here because they are used for filtering and for computing new variables later on.

In [ ]:
# Convert the dates ('Mon-Year' format, e.g. 'Dec-2015') to datetime
for col in ['fecha_concesion', 'fecha_primer_credito']:
    prestamos[col] = pd.to_datetime(prestamos[col], format='%b-%Y')

# Resulting type of each column
prestamos.dtypes.to_frame('tipo')

**What is done:** the two dates are converted to datetime and the data type (`tipo`) of each column is shown.

**What we see:** the dates are now `datetime64`. Some variables are still text (`str`), such as `num_cuotas` (`" 36 months"`) or `antigüedad_empleo` (`"10+ years"`); they will be handled in notebook 02.

<a id="s13"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">13 · D12 · TRAINING / VALIDATION SPLIT</span>

</div>

> **Decision D12 · Random 70 % training / 30 % validation split, stratified by default and with a fixed seed (`random_state=42`).**
> - **Stratifying** guarantees the same default rate in both sets.
> - **The fixed seed** makes anyone running the notebook get exactly the same split.
> - **Alternative considered:** *out-of-time* validation (train on the older years and validate on 2015). It is more realistic, but 60-month loans only go up to 2013 (D5), so the validation set would only contain 36-month loans and would not be comparable. It can be done as an additional analysis.

In [ ]:
# Default flag, only for stratifying (the final target is built in notebook 04)
impago = prestamos['estado'].isin(ESTADOS_IMPAGO)

# Stratified, reproducible split
trabajo, val = train_test_split(
    prestamos,
    test_size=PROP_VALIDACION,   # 30 % for validation
    stratify=impago,             # same share of defaults in both sets
    random_state=SEMILLA,        # same split on every run
)

# Check: no loan is in both sets
assert not set(trabajo['id_cliente']) & set(val['id_cliente']), "There are loans in both sets"

# Size and default rate of each set
pd.DataFrame({
    'n_prestamos': [len(trabajo), len(val)],
    'tasa_impago': [trabajo['estado'].isin(ESTADOS_IMPAGO).mean(),
                    val['estado'].isin(ESTADOS_IMPAGO).mean()],
}, index=['training', 'validation'])

**What is done:** the data is split into training (70 %) and validation (30 %), stratifying by default and with a fixed seed. We check that no loan is in both sets.

**What we see:** **443,469** training loans and **190,059** validation loans, with the **same default rate (14.8 %)** (`tasa_impago`) in both thanks to stratification.

<a id="s14"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">14 · SAVE THE DATASETS</span>

</div>

In [ ]:
# Create the destination folders if they do not exist
RUTA_ENTRENAMIENTO.parent.mkdir(parents=True, exist_ok=True)
RUTA_VALIDACION.parent.mkdir(parents=True, exist_ok=True)

# Save in pickle format (keeps pandas types; it is what the following notebooks read)
trabajo.to_pickle(RUTA_ENTRENAMIENTO)
val.to_pickle(RUTA_VALIDACION)
print(f"✅ Training:   {RUTA_ENTRENAMIENTO} ({len(trabajo):,} rows)")
print(f"✅ Validation: {RUTA_VALIDACION} ({len(val):,} rows)")

**What is done:** both sets are saved in *pickle* format, which keeps pandas data types.

**What we see:** the files `train.pkl` and `validacion.pkl` are saved in `02_datos`, ready for notebook 02.

<a id="s15"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">15 · SUMMARY: ROW FUNNEL</span>

</div>

In [ ]:
# Rows remaining after each decision
embudo['7. Training (70 %)'] = len(trabajo)
embudo['8. Validation (30 %)'] = len(val)

tabla_embudo = pd.Series(embudo).to_frame('n_filas')
tabla_embudo['% of the original'] = tabla_embudo['n_filas'] / embudo['1. Original file'] * 100
tabla_embudo

**What is done:** table with the rows (`n_filas`) remaining after each decision and their % of the original file.

**What we see:** the largest losses come from D4 (loans in progress, from 100 % to 59.6 %) and D5 (incomplete cohorts, down to 29.9 %). In the end **28 % of the file** is used (633,528 loans), split into training (19.6 %) and validation (8.4 %).

<a id="s16"></a>
<div style="background-color:#1f4e79; padding:2px 16px; border-radius:6px; margin-top:16px;">

## <span style="color:#ffffff">16 · NEXT STEPS</span>

</div>

- **02 · Data quality:** convert texts to numbers (`num_cuotas`, `antigüedad_empleo`), handle nulls with a meaning (`num_meses_desde_ult_retraso`: the borrower has never been late) and review anomalous values.
- **03 · EDA:** relationship of each variable with default.
- **04 · Transformation:** compute the length of credit history (`fecha_concesion` − `fecha_primer_credito`) and build the three targets from `estado`, `imp_amortizado` and `imp_recuperado`.
- **05–07 · Models:** PD (classification), EAD and LGD (regression).
- **08 · Pre-production:** the variables of the final model will be the ones asked for in the demo form.